
2.1 理论计算题
## 1. 计算 $p(\boldsymbol{a\mid b})$
$count(b\rightarrow a)=1$
$$
p(a|b)=\frac{1+1}{2+3}=\frac{2}{5}=\boldsymbol{0.4}
$$

## 2. 计算 $p(\boldsymbol{c\mid b})$
$count(b\rightarrow c)=1$
$$
p(c|b)=\frac{1+1}{2+3}=\frac{2}{5}=\boldsymbol{0.4}
$$

### 校验剩余概率（b→b）
$$
p(b|b)=\frac{0+1}{2+3}=\frac{1}{5}=0.2
$$
总和：$0.4+0.4+0.2=1$，概率合法。

---
### 最终结果
1. $\boldsymbol{p(a\mid b)=\dfrac{2}{5}}$
2. $\boldsymbol{p(c\mid b)=\dfrac{2}{5}}$

In [ ]:
#2.2 编程题
import string
from collections import Counter

def preprocess_text(text, n):
    # 步骤1：转小写，去除标点，仅保留字母和空格
    text_lower = text.lower()
    # 构建标点删除映射
    trans_table = str.maketrans('', '', string.punctuation)
    text_clean = text_lower.translate(trans_table)
    
    # 步骤2：按空格分词
    words = text_clean.split()
    
    # 步骤3：构建词汇表，按频率降序排序，ID从0开始
    word_count = Counter(words)
    # 先按频次降序，频次相同按单词字典序升序
    sorted_words = sorted(word_count.keys(), key=lambda x: (-word_count[x], x))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 步骤4：滑动窗口生成特征序列与标签
    features = []
    labels = []
    total_len = len(words)
    for i in range(total_len - n):
        window = words[i:i+n]
        next_word = words[i + n]
        features.append(window)
        labels.append(next_word)
    # 处理最后一个窗口，无后续词标签为None
    if total_len >= n:
        last_window = words[-n:]
        features.append(last_window)
        labels.append(None)
    
    return vocab, (features, labels)

# 题目示例测试
if __name__ == "__main__":
    test_text = "The time machine"
    vocab, (feats, labs) = preprocess_text(test_text, n=2)
    print("词汇表：", vocab)
    print("特征列表：", feats)
    print("标签列表：", labs)

词汇表： {'machine': 0, 'the': 1, 'time': 2}
特征列表： [['the', 'time'], ['time', 'machine']]
标签列表： ['machine', None]


3.1 理论计算题
## 1. 单时间步损失对 $o_t$
令 $L_t=\frac{1}{2}(o_t-y_t)^2$，$L=\sum_{t=1}^T L_t$
$$
\frac{\partial L_t}{\partial o_t}=o_t-y_t
$$

## 2. 对 $h_t$ 求导
$$
\frac{\partial L_t}{\partial h_t}
=\frac{\partial L_t}{\partial o_t}\cdot \frac{\partial o_t}{\partial h_t}
=(o_t-y_t)W_{oh}^\top
$$
记 $\displaystyle \delta_t = \frac{\partial L}{\partial h_t}$，则 $\displaystyle \delta_t = \frac{\partial L_t}{\partial h_t} + \frac{\partial L_{t+1}}{\partial h_t}$

由 $h_{t+1}=W_{hh}h_t+W_{hx}x_{t+1}$：
$$
\frac{\partial L_{t+1}}{\partial h_t}
=\frac{\partial L_{t+1}}{\partial h_{t+1}}\cdot \frac{\partial h_{t+1}}{\partial h_t}
=\delta_{t+1} W_{hh}
$$
得到**误差反向递推公式**：
$$
\boldsymbol{\delta_t = (o_t-y_t)W_{oh}^\top \;+\; \delta_{t+1} W_{hh}}
$$
边界条件：最后时刻 $\delta_{T+1}=\boldsymbol{0}$，即 $\delta_T=(o_T-y_T)W_{oh}^\top$

## 3. 对 $W_{hh}$ 求梯度
由 $h_t = W_{hh}h_{t-1}+W_{hx}x_t \implies \displaystyle \frac{\partial h_t}{\partial W_{hh}} = h_{t-1}^\top \otimes I$，矩阵形式：
$$
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^T \frac{\partial L}{\partial h_t}\cdot \frac{\partial h_t}{\partial W_{hh}}
=\boldsymbol{\sum_{t=1}^T \delta_t^\top \, h_{t-1}}
$$

### 展开所有时间步完整表达式
将 $\delta_t$ 逐层展开：
$$
\begin{align*}
\delta_t
&=(o_t-y_t)W_{oh}^\top
+(o_{t+1}-y_{t+1})W_{oh}^\top W_{hh}
+(o_{t+2}-y_{t+2})W_{oh}^\top W_{hh}^2
+\dots
+(o_T-y_T)W_{oh}^\top W_{hh}^{T-t}
\end{align*}
$$
代入梯度：
$$
\boxed{
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^T \left[
\left(\sum_{k=t}^T (o_k-y_k)W_{oh}^\top \, W_{hh}^{k-t}\right)^\top h_{t-1}
\right]
}
$$

---
# 梯度消失/爆炸成因分析
梯度递推核心项包含幂次 $\boldsymbol{W_{hh}^{k-t}}$，设矩阵 $W_{hh}$ 谱半径（最大特征值绝对值）为 $\rho$：
1. **梯度消失**
若 $\boldsymbol{\rho < 1}$，随着时间间隔 $k-t$ 增大，$W_{hh}^{k-t}$ 元素指数衰减，远距离梯度趋近于0，长依赖梯度几乎消失。

2. **梯度爆炸**
若 $\boldsymbol{\rho > 1}$，随着时间间隔 $k-t$ 增大，$W_{hh}^{k-t}$ 元素指数激增，梯度数值无限变大，参数更新震荡发散。

3. **临界稳定**
$\rho=1$ 时梯度幅度基本保持，理论稳定，但数值极易受噪声扰动。

> 本质：线性RNN反向传播存在重复乘 $W_{hh}$ 的连乘结构，特征值决定连乘收敛/发散趋势。

In [ ]:
#3.2 编程题
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单步前向传播
    :param x_t: (batch_size, input_size)
    :param h_prev: (batch_size, hidden_size)
    :param W_hx: (input_size, hidden_size)
    :param W_hh: (hidden_size, hidden_size)
    :param b_h: (hidden_size,)
    :return: h_t, z_t （z_t缓存用于反向传播）
    """
    z_t = np.matmul(x_t, W_hx) + np.matmul(h_prev, W_hh) + b_h
    h_t = np.tanh(z_t)
    return h_t, z_t


def rnn_backward(dh_next, x_t, h_prev, h_t, z_t, W_hx, W_hh):
    """
    RNN单步反向传播，计算所有梯度
    :param dh_next: dL/dh_t, shape (batch_size, hidden_size)
    :return: dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    # tanh 局部梯度
    dz_t = dh_next * (1 - h_t ** 2)

    # 输入梯度
    dx_t = np.matmul(dz_t, W_hx.T)
    # 上一时刻隐藏状态梯度
    dh_prev = np.matmul(dz_t, W_hh.T)

    # 权重梯度
    dW_hx = np.matmul(x_t.T, dz_t)
    dW_hh = np.matmul(h_prev.T, dz_t)
    # 偏置梯度，batch维度求和
    db_h = np.sum(dz_t, axis=0)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试示例
if __name__ == "__main__":
    batch_size = 2
    input_size = 3
    hidden_size = 4

    # 随机初始化输入、隐状态、权重
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)

    # 前向
    h_t, z_t = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("h_t shape:", h_t.shape)

    # 模拟上游梯度
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, x_t, h_prev, h_t, z_t, W_hx, W_hh)

    print("dx_t shape:", dx_t.shape)
    print("dh_prev shape:", dh_prev.shape)
    print("dW_hx shape:", dW_hx.shape)
    print("dW_hh shape:", dW_hh.shape)
    print("db_h shape:", db_h.shape)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (4,)


4.1 理论计算题
# 一、单层单向RNN参数量基础公式
单个单向RNN单元（权重+偏置）：
输入权重 $W_{xh}\in \mathbb{R}^{D_{in}\times H}$，隐状态权重 $W_{hh}\in \mathbb{R}^{H\times H}$，偏置 $b_h\in\mathbb{R}^H$
$$
Param_{single\_dir} = D_{in}\cdot H + H\cdot H + H
$$

双向RNN包含**前向、反向两个独立RNN**，单层双向RNN参数：
$$
Param_{bi\_1layer}=2\big(D_{in}\cdot H + H^2 + H\big)
$$

---
# 二、L层深度双向RNN分层参数
## 第1层（输入维度=$D$）
$$
P_1 = 2\big(D\cdot H + H^2 + H\big)
$$

## 第2~L层（上一层双向拼接输出维度=$2H$，作为本层输入）
每层输入维度固定为 $2H$，单层双向参数：
$$
P_{mid}=2\big(2H\cdot H + H^2 + H\big)=2\big(3H^2+H\big)
$$
中间层数总数量：$L-1$ 层
$$
P_{2\sim L}=(L-1)\cdot 2\big(3H^2+H\big)
$$

## RNN堆叠部分总参数
$$
\begin{align*}
P_{rnn}
&= P_1 + P_{2\sim L}\\
&= 2(DH+H^2+H)+2(L-1)(3H^2+H)
\end{align*}
$$

## 末尾输出全连接层
双向RNN最后一层输出拼接维度为 $2H$，输出维度 $O$，含权重+偏置：
$$
P_{out}=2H\cdot O + O
$$

---
# 三、模型总参数表达式
$$
\begin{aligned}
P_{total}
&=\boldsymbol{2\left(DH+H^2+H\right)+2(L-1)\left(3H^2+H\right)+O(2H+1)}
\end{aligned}
$$

### 展开整理
$$
P_{total}=2DH + \big(6L-4\big)H^2 + 2LH + 2HO + O
$$


In [ ]:
#4.2 编程题
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 定义双向单层RNN，batch_first=False匹配输入shape (seq_len, batch, input_dim)
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=False
        )
        self.hidden_dim = hidden_dim

    def forward(self, X):
        """
        :param X: 输入序列 shape = (seq_len, batch_size, input_dim)
        :return:
            outputs: 每个时间步拼接隐状态 (seq_len, batch, 2*hidden_dim)
            final_hidden: 最后时间步拼接隐状态 (batch, 2*hidden_dim)
        """
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (num_layers*2, batch, hidden_dim)
        outputs, h_n = self.rnn(X)

        # 拼接前向最后隐状态、反向最后隐状态
        h_forward = h_n[0, :, :]
        h_backward = h_n[1, :, :]
        final_hidden = torch.cat([h_forward, h_backward], dim=-1)

        return outputs, final_hidden


# 测试示例
if __name__ == "__main__":
    seq_len = 8
    batch_size = 4
    input_dim = 5
    hidden_dim = 6

    encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim)
    # 构造输入 (seq_len, batch, input_dim)
    x = torch.randn(seq_len, batch_size, input_dim)
    all_hidden, last_hidden = encoder(x)

    print("逐时间步拼接隐状态 shape:", all_hidden.shape)
    # 预期 torch.Size([8, 4, 12])
    print("最终拼接隐状态 shape:", last_hidden.shape)
    # 预期 torch.Size([4, 12])

逐时间步拼接隐状态 shape: torch.Size([8, 4, 12])
最终拼接隐状态 shape: torch.Size([4, 12])


5.1 理论计算题
# 一、Skip-Gram 负采样（Negative Sampling）对数似然目标函数推导
## 1. 基础概率（Sigmoid 表示共现概率）
对于中心词 $w_c$ 向量 $\boldsymbol{v}_c$，上下文正样本词 $w_o$ 向量 $\boldsymbol{u}_o$：
$$
P(D=1\mid w_c,w_o)=\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o)=\frac{1}{1+e^{-\boldsymbol{v}_c^\top \boldsymbol{u}_o}}
$$
$D=1$ 表示两个词真实共现；$D=0$ 表示两个词不共现（负样本），则：
$$
P(D=0\mid w_c,w_n)=1-\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k})
$$

## 2. 单组正负样本对数似然
给定1个正样本 $w_o$，$K$ 个负样本 $w_{n_1},w_{n_2},\dots,w_{n_K}$，**最大化对数似然**：
$$
\log P=\log\big[\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o)\big]+\sum_{k=1}^K \log\big[1-\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k})\big]
$$
损失函数为**负对数似然（最小化损失等价最大化似然）**：
$$
\mathcal{L}=-\log\big[\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o)\big]-\sum_{k=1}^K \log\big[1-\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k})\big]
$$

## 3. 完整目标函数（全局所有中心-上下文对）
遍历全部中心词 $w_c$、其全部上下文词 $w_o$，整体目标：
$$
\mathcal{J}=-\sum_{w_c}\sum_{w_o\in Context(w_c)}
\left[
\log\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o)
+\sum_{k=1}^K \log\big(1-\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k})\big)
\right]
$$

---
# 二、负样本采样（噪声分布采样规则）
## 1. 采样分布定义
负样本不随机均匀抽取，采用**词频 3/4 幂次分布（Unigram 幂分布）**，是Word2Vec标准噪声分布：
设词典总词数为 $V$，词 $w$ 出现频次为 $f(w)$，总词频次 $N=\sum_{w\in V}f(w)$，采样概率：
$$
P_n(w)=\frac{f(w)^{3/4}}{\sum_{w'\in V}f(w')^{3/4}}
$$

## 2. 采样约束
1. 禁止采样**正样本词 $w_o$、中心词 $w_c$**，避免正负样本重复；
2. 每次针对一组 $(w_c,w_o)$，独立从 $P_n(w)$ 中抽取 $K$ 个不同单词作为负样本；
3. 高频词更容易被选为负样本，符合语言统计规律，提升词向量训练效果。

## 3. 采样目的
相比原始Hierarchical Softmax、全词典分类，负采样仅需更新 $K+1$ 个词向量，极大降低计算复杂度，适合大规模词典训练。

In [ ]:
#5.2 编程题
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices_batch, target_indices, W, W_out):
    """
    CBOW前向传播 + 交叉熵损失计算（完整Softmax，无负采样）
    :param context_indices_batch: 批次上下文索引，shape [batch_size, context_size]
    :param target_indices: 批次中心词目标索引，shape [batch_size]
    :param W: 输入嵌入矩阵 (V, d)
    :param W_out: 输出权重矩阵 (d, V)
    :return: 批次平均交叉熵损失标量
    """
    batch_size, context_size = context_indices_batch.shape

    # 1. 查表得到所有上下文词向量: [batch, context_size, d]
    context_embeds = W[context_indices_batch]

    # 2. 求均值得到隐藏层向量 h: [batch, d]
    h = torch.mean(context_embeds, dim=1)

    # 3. 计算输出logits: h @ W_out -> [batch, V]
    logits = torch.matmul(h, W_out)

    # 4. 交叉熵损失（内部自带log_softmax，等价完整Softmax损失）
    loss = F.cross_entropy(logits, target_indices)

    return loss


# 测试示例
if __name__ == "__main__":
    V = 100    # 词汇表大小
    d = 16     # 嵌入维度
    batch_size = 4
    context_size = 2

    # 初始化权重
    W = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)

    # 构造输入：每个样本2个上下文词索引
    context_batch = torch.randint(low=0, high=V, size=(batch_size, context_size))
    # 中心词目标索引
    target_batch = torch.randint(low=0, high=V, size=(batch_size,))

    loss_val = cbow_forward_loss(context_batch, target_batch, W, W_out)
    print(f"CBOW 批次平均损失值: {loss_val.item():.4f}")
    

CBOW 批次平均损失值: 6.0647


6.1 理论计算题
# 缩放点积注意力分步推导（无掩码）
已知：
$$
Q\in\mathbb{R}^{2\times4},\quad K\in\mathbb{R}^{3\times4},\quad V\in\mathbb{R}^{3\times5},\quad d_k=4
$$
缩放点积注意力公式：
$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

## 步骤1：计算 $K^\top$
$K$ 形状 $(3,4)$，转置 $K^\top$ 形状为 $\boldsymbol{(4,3)}$

## 步骤2：原始点积 $QK^\top$
$Q(2,4) \times K^\top(4,3) \implies$ 结果矩阵形状：$\boldsymbol{(2,3)}$

## 步骤3：缩放得分矩阵 $\mathrm{score}=\dfrac{QK^\top}{\sqrt{d_k}}$
$$
\sqrt{d_k}=\sqrt{4}=2
$$
$$
\mathrm{Score}=\frac{QK^\top}{2}\in\mathbb{R}^{2\times 3}
$$

## 步骤4：按行 Softmax（无掩码）
对得分矩阵**每一行独立做 Softmax**，得到注意力权重矩阵 $\mathrm{AttnWeight}\in\mathbb{R}^{2\times3}$
$$
\mathrm{AttnWeight}_{i,j}
=\frac{\exp(\mathrm{Score}_{i,j})}{\sum_{m=1}^3\exp(\mathrm{Score}_{i,m})}
$$

## 步骤5：加权求和乘 $V$ 得到输出
$\mathrm{AttnWeight}(2,3) \times V(3,5)$
最终注意力输出矩阵形状：$\boldsymbol{(2\times 5)}$

---
# 维度总结
1. $QK^\top$：$(2,3)$
2. 缩放得分矩阵：$(2,3)$
3. Softmax注意力权重：$(2,3)$
4. 最终注意力输出：$\boldsymbol{(2,5)}$

如需代入具体数值完整演算，我可以举一组具体矩阵从头算出数值结果。

In [ ]:
#6.2 编程题
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_heads = 2
        self.d_model = 4
        self.d_k = self.d_model // self.num_heads  # d_k = 2

        # Q/K/V 整体投影层 (d_model -> d_model)
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)

        # 输出融合线性层
        self.w_o = nn.Linear(self.d_model, self.d_model)

    def scaled_dot_product_attn(self, q, k, v):
        """
        单头缩放点积注意力
        q,k,v: (num_heads, seq_len, batch, d_k)
        """
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn_weight = F.softmax(attn_score, dim=-1)
        out = torch.matmul(attn_weight, v)
        return out

    def forward(self, X):
        seq_len, batch, _ = X.shape

        # 1. 线性投影得到 Q, K, V
        Q = self.w_q(X)
        K = self.w_k(X)
        V = self.w_v(X)

        # 2. 分头：reshape 拆成多头
        # (seq_len, batch, d_model) -> (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 0, 1, 3)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 0, 1, 3)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 0, 1, 3)

        # 3. 每个头计算注意力
        head_out = self.scaled_dot_product_attn(Q, K, V)

        # 4. 多头拼接恢复 d_model
        head_out = head_out.permute(1, 2, 0, 3).contiguous()
        concat = head_out.view(seq_len, batch, self.d_model)

        # 5. 最终线性层输出
        output = self.w_o(concat)
        return output


# 测试代码
if __name__ == "__main__":
    mha = MultiHeadAttention()
    seq_len = 5
    batch_size = 3
    d_model = 4
    X = torch.randn(seq_len, batch_size, d_model)
    out = mha(X)
    print("输入形状:", X.shape)
    print("输出形状:", out.shape)
    # 输出形状与输入完全一致: torch.Size([5, 3, 4])

输入形状: torch.Size([5, 3, 4])
输出形状: torch.Size([5, 3, 4])
